# m21b · Fine-tuning hands-on: LoRA en Colab

Este notebook es la **práctica** del [m21 · Fine-tuning vs RAG](../../curso-langchain.html). El m21 te dio la *decisión*; aquí **afinas un modelo de verdad** con **LoRA/QLoRA**.

> ⚠️ **Fuera del gate offline del curso, a propósito** (ver `docs/adr/0003-notebook-lora-fuera-del-gate.md`). Necesita **GPU** y descarga varios GB, así que **no** corre en la CI ni se instala en `pyproject.toml`. Córrelo en **Google Colab** con entorno de ejecución **GPU** (menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*), no en tu máquina.

**Qué vas a hacer:** cargar un modelo base pequeño en 4 bits (QLoRA), ponerle unos adaptadores LoRA (entrenas ~1% de los pesos), afinarlo con un mini-dataset de estilo y comprobar el cambio de *comportamiento*. Recuerda la regla del m21: LoRA enseña **la forma** (tono, formato), no conocimiento nuevo — para datos, RAG.

## 1 · Entorno (instala en la sesión de Colab, no en tu curso)

In [ ]:
# Estas libs viven SOLO en la sesión de Colab. No van al pyproject del curso.
!pip install -q transformers peft datasets accelerate bitsandbytes trl

import torch
assert torch.cuda.is_available(), "Activa la GPU: Entorno de ejecución → Cambiar tipo → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2 · Cargar el modelo base en 4 bits (esto es la *Q* de QLoRA)

Cuantizar el base a 4 bits es lo que hace que un modelo de miles de millones de parámetros **quepa en la GPU gratuita** de Colab. Sobre ese base congelado pondremos los adaptadores.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Un modelo base pequeño. VERIFICA el id actual en huggingface.co (cambian seguido);
# cualquier modelo instruct de <2B sirve para practicar en la GPU gratuita.
MODELO_BASE = "Qwen/Qwen2.5-0.5B-Instruct"

qlora_4bit = BitsAndBytesConfig(
    load_in_4bit=True,                    # la Q de QLoRA: base en 4 bits
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE, quantization_config=qlora_4bit, device_map="auto",
)
print("Base cargado y cuantizado a 4 bits.")

## 3 · Poner los adaptadores LoRA (entrenas ~1% de los pesos)

LoRA congela el modelo original y añade unas matrices pequeñas (`r` filas) en las capas de atención. `print_trainable_parameters()` te enseña la magia: entrenas una fracción minúscula del total.

In [ ]:
from peft import LoraConfig, get_peft_model

lora = LoraConfig(
    r=8,                  # el "ancho" del adaptador: más grande = más capacidad y más memoria
    lora_alpha=16,        # escala de la actualización
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],   # dónde se inyecta (capas de atención)
    task_type="CAUSAL_LM",
)

modelo = get_peft_model(modelo, lora)
modelo.print_trainable_parameters()   # p. ej. 'trainable: 0.5M || all: 500M || 0.1%'

## 4 · El dataset (formato chat — el mismo del m21)

Aquí enseñamos un **comportamiento**: responder siempre corto y en tono de vendedor amable. La *calidad* del dataset ES la calidad del modelo afinado; en un caso real querrás cientos o miles de ejemplos buenos, no tres.

In [ ]:
from datasets import Dataset

SISTEMA = "Eres un asesor de ventas: responde en UNA frase, claro y amable."
pares = [
    ("¿Tienen envíos?",        "¡Claro! Enviamos a todo el país en 24-48 h. 😊"),
    ("¿Aceptan Yape?",         "Sí, aceptamos Yape, Plin y tarjeta sin recargo."),
    ("¿Puedo devolver algo?",  "Por supuesto: tienes 7 días para cambios o devolución."),
]

def a_chat(usuario, respuesta):
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SISTEMA},
         {"role": "user", "content": usuario},
         {"role": "assistant", "content": respuesta}],
        tokenize=False,
    )

dataset = Dataset.from_dict({"text": [a_chat(u, r) for u, r in pares]})
print(dataset[0]["text"][:200])

## 5 · Entrenar (SFTTrainer de `trl`)

Con 3 ejemplos y pocas épocas esto es un *juguete* para ver el mecanismo end-to-end en un par de minutos. Fíjate en que solo se actualizan los adaptadores LoRA.

In [ ]:
from trl import SFTConfig, SFTTrainer

cfg = SFTConfig(
    output_dir="lora-vendedor",
    num_train_epochs=10,        # juguete: pocos ejemplos, varias vueltas
    per_device_train_batch_size=1,
    learning_rate=2e-4,
    logging_steps=5,
    report_to="none",
)
trainer = SFTTrainer(model=modelo, train_dataset=dataset, args=cfg)
trainer.train()

## 6 · Guardar los adaptadores y probar el cambio de comportamiento

Guardas **solo los adaptadores** (unos MB), no el modelo entero. En producción los cargas encima del base con `PeftModel.from_pretrained(base, ruta)`.

In [ ]:
modelo.save_pretrained("lora-vendedor")   # solo los adaptadores (~MB, no GB)

prompt = tokenizer.apply_chat_template(
    [{"role": "system", "content": SISTEMA},
     {"role": "user", "content": "¿Hacen factura?"}],
    tokenize=False, add_generation_prompt=True,
)
entradas = tokenizer(prompt, return_tensors="pt").to(modelo.device)
salida = modelo.generate(**entradas, max_new_tokens=40)
print(tokenizer.decode(salida[0], skip_special_tokens=True))

## Qué acabas de aprender

- **QLoRA** = base cuantizado a 4 bits + adaptadores LoRA: así un modelo grande cabe en la GPU gratuita.
- Entrenaste **~1% de los pesos** y aun así cambiaste el *comportamiento* (tono, longitud).
- Los adaptadores pesan **MB**, no GB: se guardan y se cargan encima del base.
- **La regla del m21 sigue en pie:** afinaste la *forma*, no metiste conocimiento. Para datos que cambian (precios, stock, FAQ), sigue siendo **RAG**.

Vuelve al [m21](../../curso-langchain.html) para la decisión completa, y al m18b para el coste de servir tu modelo afinado.